In [ ]:
!pip install -q keras-tuner

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
(full_train_images, full_train_labels), (test_images, test_labels) = keras.datasets.fashion_mnist.load_data()
full_train_images = full_train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

train_images, val_images = full_train_images[:50000], full_train_images[50000:]
train_labels, val_labels = full_train_labels[:50000], full_train_labels[50000:]
class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for axis, index in zip(axes.flatten(), range(16)):
    axis.imshow(train_images[index], cmap="gray")
    axis.set_title(class_names[train_labels[index]], fontsize=9)
    axis.axis("off")
plt.show()

In [ ]:
def build_mlp(hidden_units=128, hidden_layers=2, learning_rate=1e-3, dropout_rate=0.0, activation="relu"):
    model = keras.Sequential([layers.Input(shape=(28, 28)), layers.Flatten()])
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def run_experiment(epochs=8, batch_size=128, **settings):
    keras.utils.set_random_seed(42)
    model = build_mlp(**settings)
    start_time = time.time()
    history = model.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=epochs, batch_size=batch_size, verbose=0)
    return {
        "history": history.history,
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "final_val_loss": history.history["val_loss"][-1],
        "parameters": model.count_params(),
        "seconds": time.time() - start_time
    }

def plot_sweep(results, setting_name, log_scale=False):
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    for value, result in results.items():
        axes[0].plot(result["history"]["val_loss"], label=str(value))
        axes[1].plot(result["history"]["val_accuracy"], label=str(value))
    axes[0].set_title(f"validation loss by {setting_name}")
    axes[1].set_title(f"validation accuracy by {setting_name}")
    for axis in axes[:2]:
        axis.set_xlabel("epoch")
        axis.legend(title=setting_name, fontsize=8)
    values = list(results.keys())
    axes[2].plot([str(value) for value in values] if not log_scale else values, [results[value]["best_val_accuracy"] for value in values], marker="o")
    if log_scale:
        axes[2].set_xscale("log")
    axes[2].set_title(f"best validation accuracy vs {setting_name}")
    axes[2].set_xlabel(setting_name)
    plt.show()

def sweep_table(results, setting_name):
    return pd.DataFrame([
        {setting_name: value, "best_val_accuracy": round(result["best_val_accuracy"], 4), "final_val_loss": round(result["final_val_loss"], 4), "parameters": result["parameters"], "seconds": round(result["seconds"], 1)}
        for value, result in results.items()
    ])

In [ ]:
learning_rate_results = {}
for learning_rate in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]:
    learning_rate_results[learning_rate] = run_experiment(learning_rate=learning_rate)
    print(f"learning rate {learning_rate:<8} best val accuracy {learning_rate_results[learning_rate]['best_val_accuracy']:.4f}")

plot_sweep(learning_rate_results, "learning rate", log_scale=True)
display(sweep_table(learning_rate_results, "learning rate"))

In [ ]:
hidden_unit_results = {}
for hidden_units in [16, 32, 64, 128, 256, 512]:
    hidden_unit_results[hidden_units] = run_experiment(hidden_units=hidden_units)
    print(f"hidden units {hidden_units:<4} best val accuracy {hidden_unit_results[hidden_units]['best_val_accuracy']:.4f}")

plot_sweep(hidden_unit_results, "hidden units")
display(sweep_table(hidden_unit_results, "hidden units"))

In [ ]:
depth_results = {}
for hidden_layers in [1, 2, 3, 4, 6]:
    depth_results[hidden_layers] = run_experiment(hidden_layers=hidden_layers, hidden_units=256)
    print(f"hidden layers {hidden_layers} best val accuracy {depth_results[hidden_layers]['best_val_accuracy']:.4f}")

plot_sweep(depth_results, "hidden layers")
display(sweep_table(depth_results, "hidden layers"))

In [ ]:
batch_size_results = {}
for batch_size in [16, 64, 256, 1024]:
    batch_size_results[batch_size] = run_experiment(batch_size=batch_size, hidden_units=256)
    print(f"batch size {batch_size:<5} best val accuracy {batch_size_results[batch_size]['best_val_accuracy']:.4f}  time {batch_size_results[batch_size]['seconds']:.1f}s")

plot_sweep(batch_size_results, "batch size", log_scale=True)
display(sweep_table(batch_size_results, "batch size"))

In [ ]:
activation_results = {}
for activation in ["sigmoid", "tanh", "relu", "elu", "gelu"]:
    activation_results[activation] = run_experiment(activation=activation, hidden_units=256)
    print(f"activation {activation:<8} best val accuracy {activation_results[activation]['best_val_accuracy']:.4f}")

plot_sweep(activation_results, "activation")

dropout_results = {}
for dropout_rate in [0.0, 0.1, 0.2, 0.3, 0.5]:
    dropout_results[dropout_rate] = run_experiment(dropout_rate=dropout_rate, hidden_units=512, epochs=12)
    print(f"dropout {dropout_rate:<4} best val accuracy {dropout_results[dropout_rate]['best_val_accuracy']:.4f}")

plot_sweep(dropout_results, "dropout rate")

In [ ]:
grid_rows = []
for learning_rate in [3e-4, 1e-3, 3e-3]:
    for hidden_units in [64, 256, 512]:
        result = run_experiment(learning_rate=learning_rate, hidden_units=hidden_units)
        grid_rows.append({"learning_rate": learning_rate, "hidden_units": hidden_units, "best_val_accuracy": result["best_val_accuracy"]})
        print(f"lr {learning_rate:<7} units {hidden_units:<4} -> {result['best_val_accuracy']:.4f}")

grid_frame = pd.DataFrame(grid_rows)
plt.figure(figsize=(7, 5))
sns.heatmap(grid_frame.pivot(index="learning_rate", columns="hidden_units", values="best_val_accuracy"), annot=True, fmt=".4f", cmap="viridis")
plt.title("Grid search: learning rate x hidden units")
plt.show()

In [ ]:
def build_tunable_cnn(hp):
    model = keras.Sequential([layers.Input(shape=(28, 28, 1))])
    kernel_size = hp.Choice("kernel_size", [3, 5])
    for block in range(hp.Int("conv_blocks", 1, 3)):
        model.add(layers.Conv2D(hp.Choice(f"filters_{block}", [32, 64, 128]), kernel_size, padding="same", activation="relu"))
        if hp.Boolean("batch_norm"):
            model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D())
    model.add(layers.Flatten())
    model.add(layers.Dense(hp.Int("dense_units", 64, 256, step=64), activation="relu"))
    model.add(layers.Dropout(hp.Float("dropout", 0.0, 0.5, step=0.1)))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

tuner = kt.RandomSearch(
    build_tunable_cnn,
    objective="val_accuracy",
    max_trials=12,
    executions_per_trial=1,
    directory="tuning_results",
    project_name="fashion_cnn",
    overwrite=True,
    seed=42
)
tuner.search_space_summary()

In [ ]:
tuner.search(
    train_images[..., None], train_labels,
    validation_data=(val_images[..., None], val_labels),
    epochs=6,
    batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2)]
)

In [ ]:
trial_rows = []
for trial in tuner.oracle.get_best_trials(num_trials=12):
    row = dict(trial.hyperparameters.values)
    row["val_accuracy"] = trial.score
    trial_rows.append(row)
trial_frame = pd.DataFrame(trial_rows).sort_values("val_accuracy", ascending=False)
display(trial_frame)

best_hyperparameters = tuner.get_best_hyperparameters(1)[0]
print("best hyperparameters:")
for name, value in best_hyperparameters.values.items():
    print(f"  {name}: {value}")

In [ ]:
keras.utils.set_random_seed(42)
best_cnn = tuner.hypermodel.build(best_hyperparameters)
best_cnn_history = best_cnn.fit(
    train_images[..., None], train_labels,
    validation_data=(val_images[..., None], val_labels),
    epochs=20,
    batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)]
)

keras.utils.set_random_seed(42)
baseline_mlp = build_mlp()
baseline_mlp.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=20, batch_size=128, verbose=0, callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)])

baseline_test_accuracy = baseline_mlp.evaluate(test_images, test_labels, verbose=0)[1]
tuned_test_accuracy = best_cnn.evaluate(test_images[..., None], test_labels, verbose=0)[1]
print(f"default MLP test accuracy: {baseline_test_accuracy:.4f}")
print(f"tuned CNN test accuracy:   {tuned_test_accuracy:.4f}")

In [ ]:
summary_rows = []
for setting_name, results in [("learning rate", learning_rate_results), ("hidden units", hidden_unit_results), ("hidden layers", depth_results), ("batch size", batch_size_results), ("activation", activation_results), ("dropout rate", dropout_results)]:
    best_value = max(results, key=lambda value: results[value]["best_val_accuracy"])
    worst_value = min(results, key=lambda value: results[value]["best_val_accuracy"])
    summary_rows.append({
        "hyperparameter": setting_name,
        "best value": best_value,
        "best val accuracy": round(results[best_value]["best_val_accuracy"], 4),
        "worst value": worst_value,
        "worst val accuracy": round(results[worst_value]["best_val_accuracy"], 4)
    })
display(pd.DataFrame(summary_rows))

plt.figure(figsize=(8, 4))
plt.plot(best_cnn_history.history["accuracy"], label="train")
plt.plot(best_cnn_history.history["val_accuracy"], label="validation")
plt.title("Tuned CNN training")
plt.xlabel("epoch")
plt.legend()
plt.show()